# Offline replay activity and following behavior in different motivational needs (reported in Carey et al.)

### Import

In [ ]:
import os, sys, pickle, warnings
import numpy as np
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

base_path = os.path.sep.join(os.path.abspath("__file__").split(os.path.sep)[:-2])
sys.path.insert(0, os.path.join(base_path, "code/functions"))

from Tmaze_functions import *
from stat_functions import *
from plots import create_annot_nonzero

plot_path = os.path.join(data_path, "plots")
os.makedirs(plot_path, exist_ok=True)

save_plot = True

In [2]:
potentiated_states   = list(range(7))     # 0-indexed: 0..6  (used for weight scaling)
unpotentiated_states = list(range(7, num_state_total)) # 0-indexed: 7, 8, 9
potentiation_factor  = 1.1  # +10 % on existing synapses

region_labels = ["stem", "left", "right"]
stem_states = np.arange(3)
left_state  = [3, 4, 5]   # 0-indexed; 1-indexed: 4, 5, 6
right_state = [7, 8, 9]   # 0-indexed; 1-indexed: 8, 9, 10

cue_state = start_ID

sim_duration = 20000    # ms

trial_number   = 10
trial_dirs = [os.path.join(data_path, "trial%d" % i) for i in range(trial_number)]

# ── condition files/labels (control vs. potentiated), shared by the raster (below) and
# per-region replay-count cells ──────────────────────────────────────────────
cond_label = ["Control", "Potentiated"]
condition_fname  = ["CA3_replay_ctrl_cue1.npz", "CA3_replay_potentiated_cue1.npz"]

PF_dict = load_PF_starts()

## Offline Replay with Selective Potentiation of States 1–7

### SFig. 6b, Juxtaposed raster: potentiated (top) vs. control (bottom)

In [4]:
targ_trial = 2

fig, axes = plt.subplots(
    2, 1,
    figsize=(12, 3),
    sharey=True,
    gridspec_kw={"hspace": 0.06, "wspace": 0.05},
)

trial_idx = targ_trial; exp_dir = trial_dirs[trial_idx]

place_cell_ID_list = generate_place_cell_ID_list(np.array(list(PF_dict.keys()), dtype=int), np.array(list(PF_dict.values())))
neuron_to_state = -np.ones(num_CA3_neurons, dtype=int)
for s, ids in enumerate(place_cell_ID_list):
    neuron_to_state[ids] = s

for row, fname in enumerate(condition_fname):
    ax = axes[row]

    d          = np.load(os.path.join(exp_dir, fname), allow_pickle=True)
    spike_t    = d["spike_times_CA3_PC"]           # ms
    spike_nids = d["spiking_neurons_CA3_PC"].astype(int)
    states_v   = neuron_to_state[spike_nids]

    for s in range(num_state_total):
        mask = states_v == s
        if mask.any():
            ax.scatter(
                spike_t[mask], spike_nids[mask],
                s=0.3, c=state_color(s, left_state, right_state), rasterized=True, marker='.',
                zorder=2 if s in left_state + right_state else 1,
            )

    # state boundary lines
    for y0 in np.arange(num_state_total)*cell_per_unit:
        ax.axhline(y0, color="black", linewidth=0.4, linestyle="--", alpha=0.35)

    ax.set_xlim(0, sim_duration)
    ax.set_ylim(0, num_CA3_neurons)
    ax.tick_params(axis="x", labelsize=6)

    if row == 0:
        ax.set_title(f"E{trial_idx}", fontsize=9, pad=2)
        ax.set_xticklabels([])
    else:
        ax.set_xlabel("Time (ms)", fontsize=7)

    if trial_idx == 0:
        ax.set_ylabel(cond_label[row] + "\nneuron ID", fontsize=8)
    else:
        ax.tick_params(labelleft=False)

ytick_pos = np.arange(num_state_total)*cell_per_unit+cell_per_unit//2

for row in range(2):
    ax0 = axes[row]
    ax0.set_yticks(ytick_pos)
    ax0.set_yticklabels([f"S{s+1}" for s in range(num_state_total)], fontsize=6)
    for tick, s in zip(ax0.get_yticklabels(), range(num_state_total)):
        tick.set_color(state_color(s, left_state, right_state))

plt.show()
plt.savefig(os.path.join(plot_path, "Carey_juxtaposed_rasters_trial%d.pdf" % targ_trial),
            bbox_inches="tight")


### SFig. 6c and d,  Average firing rate: potentiated pathway vs. right arm

In [ ]:
# ── Figure: average firing rate comparison (mirrors wu_offline_fatigued.ipynb cell 13) ──
from scipy import stats

fig, axs = plt.subplots(1, 2, figsize=(5, 3))

left_rate_all   = np.zeros(trial_number)   # left arm + stem (states 1-7) rate — potentiated run
right_rate_all = np.zeros(trial_number)   # right arm (states 8-10) rate  — potentiated run

# ── Panel 1: left arm+stem vs. right arm (potentiated run only) ─────────────
ax = axs[0]
for trial_idx in range(trial_number):
    r = np.load(os.path.join(exp_dir, fname), allow_pickle=True)
    spike_nids = r["spiking_neurons_CA3_PC"].astype(int)

    place_cell_ID_list = generate_place_cell_ID_list(np.array(list(PF_dict.keys()), dtype=int), np.array(list(PF_dict.values())))
    counts = np.array([np.isin(spike_nids, place_cell_ID_list[s]).sum() for s in range(num_state_total)])

    left_rate_all[trial_idx]   = counts[left_state].sum() / (cell_per_unit * len(left_state)   * sim_duration / 1000)
    right_rate_all[trial_idx] = counts[right_state].sum() / (cell_per_unit * len(right_state) * sim_duration / 1000)

jitter = 0.04 * (np.random.default_rng(7).random(trial_number) - 0.5)
ax.scatter(np.zeros(trial_number) + jitter, right_rate_all, color="steelblue",      s=40, zorder=3)
ax.scatter(np.ones(trial_number)  + jitter, left_rate_all,   color="tomato", s=40, zorder=3)
for i in range(trial_number):
    ax.plot([jitter[i], 1 + jitter[i]], [right_rate_all[i], left_rate_all[i]],
            color="lightgrey", linewidth=0.8, zorder=1)
ax.errorbar([0, 1],
            [right_rate_all.mean(), left_rate_all.mean()],
            yerr=[right_rate_all.std() / np.sqrt(trial_number),
                  left_rate_all.std()   / np.sqrt(trial_number)],
            fmt="D", color="black", markersize=8, capsize=5, zorder=5)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Right arm", "Left arm"])
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(0, 3); ax.set_yticks(np.arange(0, 3.1, 1))
ax.set_ylabel("Firing rate")
ax.set_title("Left arm vs. right arm (potentiated run)", fontsize=10)

ax = axs[1]

left_rate_ctrl = np.full(trial_number, np.nan)
for trial_idx in range(trial_number):
    trial_dir = trial_dirs[trial_idx]
    ctrl_path = os.path.join(trial_dir, "CA3_replay_ctrl_cue1.npz")

    ctrl_data  = np.load(ctrl_path, allow_pickle=True)
    spike_nids = ctrl_data["spiking_neurons_CA3_PC"].astype(int)

    place_cell_ID_list = generate_place_cell_ID_list(np.array(list(PF_dict.keys()), dtype=int), np.array(list(PF_dict.values())))
    counts = np.array([np.isin(spike_nids, place_cell_ID_list[s]).sum() for s in range(num_state_total)])

    left_rate_ctrl[trial_idx] = counts[left_state].sum() / (cell_per_unit * len(left_state) * sim_duration / 1000)

ax.scatter(np.zeros(trial_number) + jitter, left_rate_ctrl, color="grey",      s=40, zorder=3)
ax.scatter(np.ones(trial_number)  + jitter, left_rate_all,  color="tomato", s=40, zorder=3)
for i in range(trial_number):
    ax.plot([jitter[i], 1 + jitter[i]], [left_rate_ctrl[i], left_rate_all[i]],
            color="lightgrey", linewidth=0.8, zorder=1)
ax.errorbar([0, 1],
            [np.nanmean(left_rate_ctrl), left_rate_all.mean()],
            yerr=[np.nanstd(left_rate_ctrl) / np.sqrt(trial_number),
                  left_rate_all.std()        / np.sqrt(trial_number)],
            fmt="D", color="black", markersize=8, capsize=5, zorder=5)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Control", "Potentiated"])
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(0, 3); ax.set_yticks(np.arange(0, 3.1, 1))
ax.set_ylabel("Firing rate")
ax.set_title("Potentiated vs. Not", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(plot_path, "tmaze_FR_comparison.svg"), format="svg",
            bbox_inches="tight")
plt.show()

# ── Stats ────────────────────────────────────────────────────────────────────
metrics = [
    ("Left arm vs. right arm (pot. run)", left_rate_all, right_rate_all),
    ("Left arm: ctrl vs. potentiated",     left_rate_ctrl,  left_rate_all),
]

for label, x, y in metrics:
    mask = ~(np.isnan(x) | np.isnan(y))
    xm, ym = x[mask], y[mask]
    t_stat, p_val = stats.ttest_rel(xm, ym)
    d_z = cohens_d_paired(xm, ym)
    ci_lo, ci_hi = ci_paired(xm, ym)
    print(f"{label:36s}: {xm.mean():.3f}±{xm.std(ddof=1):.3f}  vs  "
          f"{ym.mean():.3f}±{ym.std(ddof=1):.3f}  "
          f"t={t_stat:.3f}  p={p_val:.10f}  d_z={d_z:.3f}  CI=[{ci_lo:.3f}, {ci_hi:.3f}]")


Left arm vs. right arm (pot. run)   : 2.873±0.000  vs  0.943±0.000  t=inf  p=0.0000000000  d_z=inf  CI=[1.930, 1.930]
Left arm: ctrl vs. potentiated      : 1.038±0.335  vs  2.873±0.000  t=-17.331  p=0.0000000320  d_z=-5.481  CI=[-2.075, -1.596]


### SFig. 6e and f, Total replay count and replay direction

In [6]:
from scipy import stats
REGION_DEFS = {
    "stem":  {"states": stem_states, "axis": 0},   # 0 = row
    "left":  {"states": left_state,  "axis": 1},   # 1 = column
    "right": {"states": right_state, "axis": 1},
}

replay_region_counts = np.zeros((2, trial_number, 3), dtype=int)
replay_total_events  = np.zeros((2, trial_number), dtype=int)
merge_gap_dt         = 30

for cond_idx, (cond_label, fname) in enumerate(zip(cond_label, condition_fname)):
    print(f"\n=== {cond_label} ===")
    for trial_idx in range(trial_number):
        trial_dir = trial_dirs[trial_idx]

        d          = np.load(os.path.join(trial_dir, fname), allow_pickle=True)
        spike_t    = d["spike_times_CA3_PC"]           # ms
        spike_nids = d["spiking_neurons_CA3_PC"].astype(int)

        counts, n_bursts, intervals = count_replay_by_region(spike_t, spike_nids, sim_duration, REGION_DEFS)
        for ri, rlabel in enumerate(region_labels):
            replay_region_counts[cond_idx, trial_idx, ri] = counts[rlabel]

        n_total, merged_intervals = count_total_replay_events(intervals, merge_gap_dt)
        replay_total_events[cond_idx, trial_idx] = n_total

        print(f"  Tmaze{trial_idx}: bursts(stem/left/right)="
              f"{n_bursts['stem']}/{n_bursts['left']}/{n_bursts['right']}  "
              f"valid events — stem={counts['stem']}  left={counts['left']}  right={counts['right']}  "
              f"| merged total={n_total}")

print("\nDone.")



=== Control ===
  Tmaze0: bursts(stem/left/right)=12/10/10  valid events — stem=11  left=4  right=5  | merged total=20
  Tmaze1: bursts(stem/left/right)=10/9/13  valid events — stem=9  left=9  right=6  | merged total=24
  Tmaze2: bursts(stem/left/right)=10/11/11  valid events — stem=6  left=6  right=10  | merged total=22
  Tmaze3: bursts(stem/left/right)=12/10/12  valid events — stem=12  left=7  right=3  | merged total=22
  Tmaze4: bursts(stem/left/right)=10/8/12  valid events — stem=5  left=7  right=6  | merged total=18
  Tmaze5: bursts(stem/left/right)=13/11/11  valid events — stem=12  left=3  right=3  | merged total=18
  Tmaze6: bursts(stem/left/right)=10/10/10  valid events — stem=7  left=3  right=0  | merged total=10
  Tmaze7: bursts(stem/left/right)=11/10/10  valid events — stem=9  left=5  right=5  | merged total=19
  Tmaze8: bursts(stem/left/right)=13/10/10  valid events — stem=11  left=2  right=5  | merged total=18
  Tmaze9: bursts(stem/left/right)=4/2/3  valid events — stem=3

In [7]:
# ── Figure: total replay count & % replay including left arm (mirrors wu_offline_fatigued.ipynb cell 26) ──
from stat_functions import cohens_d_paired, ci_paired

fig, axes = plt.subplots(1, 2, figsize=(5, 3))

total_ctrl = np.maximum(replay_total_events[0], 1)
total_pot  = np.maximum(replay_total_events[1], 1)

# ── Panel 1: total replay count (merged across stem/left/right) ─────────────
ax = axes[0]
jitter = 0.04 * (np.random.default_rng(7).random(trial_number) - 0.5)
ax.scatter(np.zeros(trial_number) + jitter, total_ctrl, color="grey",      s=40, zorder=3)
ax.scatter(np.ones(trial_number)  + jitter, total_pot,  color="tomato", s=40, zorder=3)
for i in range(trial_number):
    ax.plot([jitter[i], 1 + jitter[i]], [total_ctrl[i], total_pot[i]],
            color="lightgrey", linewidth=0.8, zorder=1)
ax.errorbar([0, 1],
            [total_ctrl.mean(), total_pot.mean()],
            yerr=[total_ctrl.std() / np.sqrt(trial_number),
                  total_pot.std()  / np.sqrt(trial_number)],
            fmt="D", color="black", markersize=8, capsize=5, zorder=5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Control", "Potentiated"])
ax.set_xlim(-0.5, 1.5); ax.set_ylim(0, 40)
ax.set_ylabel("Replay count")
ax.set_title("Total replay count", fontsize=10)

# ── Panel 2: % replay including left arm, paired scatter ───────────────────
ax = axes[1]
left_frac_ctrl = np.sum(replay_region_counts[0, :, :2], axis=1) / np.sum(replay_region_counts[0], axis=1)
left_frac_pot  = np.sum(replay_region_counts[1, :, :2], axis=1) / np.sum(replay_region_counts[1], axis=1)

jitter2 = 0.1 * (np.random.default_rng(7).random(trial_number) - 0.5)
ax.scatter(np.zeros(trial_number) + jitter2, left_frac_ctrl, color="grey",      s=40, zorder=3)
ax.scatter(np.ones(trial_number)  + jitter2, left_frac_pot,  color="tomato", s=40, zorder=3)
for i in range(trial_number):
    ax.plot([jitter2[i], 1 + jitter2[i]], [left_frac_ctrl[i], left_frac_pot[i]],
            color="lightgrey", linewidth=1, zorder=1)
ax.errorbar([0, 1],
            [left_frac_ctrl.mean(), left_frac_pot.mean()],
            yerr=[left_frac_ctrl.std() / np.sqrt(trial_number),
                  left_frac_pot.std()  / np.sqrt(trial_number)],
            fmt="D", color="black", markersize=10, capsize=5, zorder=5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Control", "Potentiated"])
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.1, 1.1); ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.set_ylabel("(Left+stem) / merged total")
ax.set_title("% replay including left arm", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(plot_path, "tmaze_replay_direction_comparison.svg"), format="svg",
            bbox_inches="tight")
plt.show()

# ── Stats ────────────────────────────────────────────────────────────────────
metrics = [
    ("Total replay count (merged)", total_ctrl,                    total_pot),
    ("Stem count",                  replay_region_counts[1, :, 0],  replay_region_counts[0, :, 0]),
    ("Left-arm count",              replay_region_counts[1, :, 1],  replay_region_counts[0, :, 1]),
    ("Right-arm count",             replay_region_counts[1, :, 2],  replay_region_counts[0, :, 2]),
    ("Left-arm fraction",           left_frac_ctrl*100,                 left_frac_pot*100),
]

for label, x, y in metrics:
    mask = ~(np.isnan(x) | np.isnan(y))
    xm, ym = x[mask], y[mask]
    t_stat, p_val = stats.ttest_rel(xm, ym)
    d_z = cohens_d_paired(xm, ym)
    ci_lo, ci_hi = ci_paired(xm, ym)
    print(f"{label:28s}: Control={xm.mean():.3f}±{xm.std(ddof=1):.3f}  "
          f"Potentiated={ym.mean():.3f}±{ym.std(ddof=1):.3f}  "
          f"t={t_stat:.3f}  p={p_val:.4f}  d_z={d_z:.3f}  CI=[{ci_lo:.3f}, {ci_hi:.3f}]")


Total replay count (merged) : Control=17.700±5.579  Potentiated=22.200±5.007  t=-2.355  p=0.0429  d_z=-0.745  CI=[-8.822, -0.178]
Stem count                  : Control=17.000±2.625  Potentiated=8.500±3.136  t=7.671  p=0.0000  d_z=2.426  CI=[5.993, 11.007]
Left-arm count              : Control=2.800±3.048  Potentiated=4.800±2.394  t=-1.414  p=0.1909  d_z=-0.447  CI=[-5.199, 1.199]
Right-arm count             : Control=5.200±4.022  Potentiated=4.600±2.633  t=0.434  p=0.6748  d_z=0.137  CI=[-2.531, 3.731]
Left-arm fraction           : Control=74.932±12.811  Potentiated=80.804±9.380  t=-2.604  p=0.0286  d_z=-0.823  CI=[-10.975, -0.770]


## Hungry vs. thirsty T-maze behavior, using only the potentiated (offline-consolidated) network

In [ ]:
target_lap = 10        # online lap whose map gets offline-consolidated (matches generate_data.ipynb)
pause_state = 4         # offline replay cue-seeded at state 4 (food arm) -- matches the cached run
rest_time_ms = 5000      # Tmaze_variables.rest_time; offline replay duration

In [ ]:
CA1_activity_potentiated_all = np.zeros((trial_number, unit_gran*num_state_total, num_CA1_neurons))
w_CA1_feat_all = np.zeros((trial_number, num_CA1_neurons, num_features))

for trial_idx in range(trial_number):
    exp_dir = trial_dirs[trial_idx]

    CA1_activity_potentiated_all[trial_idx] = np.load(os.path.join(exp_dir, condition_fname[1]),allow_pickle=True)["CA1_activity"]
    w_CA1_feat_all[trial_idx] = np.load(os.path.join(exp_dir, "lap_10.npz"))["w_CA1_feat"]


In [14]:
RW_pred_state_potentiated = np.zeros((trial_number, num_state_total, 2))  # [:, :, 0]=food, [:, :, 1]=water

for tr in range(trial_number):
    pred_potentiated = CA1_activity_potentiated_all[tr] @ w_CA1_feat_all[tr]        # (n_spatial_points, num_features)

    for ss in range(num_state_total):
        sl = slice(ss*unit_gran+1, (ss+1)*unit_gran-1)   # trim window edges, matches analyze_Tmaze.ipynb
        RW_pred_state_potentiated[tr, ss] = np.max(pred_potentiated[sl, :2], axis=0)

print("RW_pred_state_potentiated shape:", RW_pred_state_potentiated.shape)

RW_pred_state_potentiated shape: (10, 10, 2)


In [15]:
MI_scenarios = {
    "hungry":  np.array([5, 1]),   # MI_food=5, MI_water=1
    "thirsty": np.array([1, 5]),   # MI_food=1, MI_water=5
}

food_state_idx  = num_state_row          # 0-indexed 3  -> 1-indexed state 4
water_state_idx = num_state_total - 1    # 0-indexed 9  -> 1-indexed state 10
end_state = [food_state_idx, water_state_idx]

print("Food outcome  = state %d (0-idx %d)" % (food_state_idx+1, food_state_idx))
print("Water outcome = state %d (0-idx %d)" % (water_state_idx+1, water_state_idx))

value_arr = {name: np.zeros((trial_number, num_state_total)) for name in MI_scenarios}
for name, mi_vec in MI_scenarios.items():
    for tr in range(trial_number):
        value_arr[name][tr] = pred_norm(RW_pred_state_potentiated[tr]) @ mi_vec

Food outcome  = state 4 (0-idx 3)
Water outcome = state 10 (0-idx 9)


In [16]:
possible_actions = np.array([[0,0],[0,-1],[0,1],[-1,0],[1,0]])  # stay, left, right, up-stem, down-stem
softmax_coeff   = 0.5
num_behav_trial = 100
total_time      = 50

transition_matrix = {name: np.zeros((trial_number, num_state_total, num_state_total)) for name in MI_scenarios}
for name in MI_scenarios:
    for tr in range(trial_number):
        transition_matrix[name][tr], _ = compute_transition_matrix(
            num_state_total, value_arr[name][tr], possible_actions,
            end_state=end_state, softmax_coeff=softmax_coeff)

### SFig. 6h, Mean transition matrix across all 10 trials, one panel per scenario

In [17]:
import seaborn as sns

In [18]:
scenario_names = ["hungry", "thirsty"]
titles = {"hungry": "Motivation for Food=5, Water=1", "thirsty": "Motivation for Food=1, Water=5"}

fig, axs = plt.subplots(1, 2, figsize=(6, 3.25), width_ratios=(4, 5))
for ii, name in enumerate(scenario_names):
    mean_tm = transition_matrix[name].mean(0)
    annot_data = create_annot_nonzero(10*mean_tm)
    is_last = (ii == len(scenario_names)-1)
    sns.heatmap(mean_tm, annot=annot_data, fmt="", annot_kws={"size": 9}, cmap="RdPu", ax=axs[ii],
                cbar=is_last, cbar_kws={"label": "10x Transition Prob."} if is_last else None)
    axs[ii].set_xticklabels(np.arange(1, num_state_total+1)); axs[ii].set_yticklabels(np.arange(1, num_state_total+1))
    axs[ii].invert_yaxis()
    axs[ii].set_xlabel("Future state")
    if ii == 0: axs[ii].set_ylabel("Current state")
    axs[ii].set_title(titles[name], fontsize=10)

fig.suptitle("Average Markov transition matrix (potentiated network)", y=1.05)
plt.tight_layout(); plt.show()

if save_plot: 
    plt.savefig(os.path.join(plot_path, "Carey__transition_matrix.svg"), format="svg", bbox_inches="tight")

### SFig. 6i, Choice-point asymmetry

In [20]:
choice_state, left_state, right_state = 6, 5, 7   # junction; left -> food arm, right -> water arm

asym = {name: transition_matrix[name][:, choice_state, left_state] - transition_matrix[name][:, choice_state, right_state]
        for name in MI_scenarios}

jitter = 0.04 * (np.random.default_rng(7).random(trial_number) - 0.5)

fig, ax = plt.subplots(figsize=(2, 3))
ax.scatter(np.zeros(trial_number) + jitter, asym["hungry"],  color="orange",   s=20, zorder=3)
ax.scatter(np.ones(trial_number)  + jitter, asym["thirsty"], color="steelblue", s=20, zorder=3)
for i in range(trial_number):
    ax.plot([jitter[i], 1 + jitter[i]], [asym["hungry"][i], asym["thirsty"][i]],
            color="lightgrey", linewidth=0.8, zorder=1)
ax.errorbar([0, 1], [asym["hungry"].mean(), asym["thirsty"].mean()],
            yerr=[asym["hungry"].std() / np.sqrt(trial_number), asym["thirsty"].std() / np.sqrt(trial_number)],
            fmt="D", color="black", markersize=7, capsize=5, zorder=5)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Hungry", "Thirsty"])
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.3,0.3)
ax.set_yticks(np.arange(-0.3, 0.31, 0.1)); ax.set_ylabel("P(left/food) - P(right/water)")
ax.set_title("Choice-point asymmetry\n(potentiated network)")
plt.tight_layout(); plt.show()

if save_plot: 
    plt.savefig(os.path.join(plot_path, "Carey__choice_asymmetry.svg"), format="svg", bbox_inches="tight")

# ── Stats ────────────────────────────────────────────────────────────────────
metrics = [
    ("Choice-point Asymmetry, Hungry, 1samp (pot. run)", asym["hungry"]),
    ("Choice-point Asymmetry, Thirsty, 1samp (pot. run)", asym["thirsty"]),
]

for label, x in metrics:
    mask = ~(np.isnan(x))
    xm = x[mask]
    t_stat, p_val = stats.ttest_1samp(xm, 0)
    d_z = cohens_d_1samp(xm, 0)
    ci_lo, ci_hi = ci_1samp(xm)
    print(f"{label:36s}: {xm.mean():.3f}±{xm.std(ddof=1):.3f}  vs  0\n"
          f"t={t_stat:.3f}  p={p_val:.3g}  d_z={d_z:.2f}  CI=[{ci_lo:.2f}, {ci_hi:.2f}]")


Choice-point Asymmetry, Hungry, 1samp (pot. run): 0.052±0.047  vs  0
t=3.451  p=0.00726  d_z=1.09  CI=[0.02, 0.09]
Choice-point Asymmetry, Thirsty, 1samp (pot. run): -0.143±0.045  vs  0
t=-10.106  p=3.28e-06  d_z=-3.20  CI=[-0.18, -0.11]


### SFig. 6j, Average final destination

In [21]:
init_seed = 135

goal_state_arr = np.zeros((len(MI_scenarios),trial_number,num_behav_trial))
goal_count = np.zeros((len(MI_scenarios),3,trial_number))

frac = {}

for vv, name in enumerate(MI_scenarios):
    for mm in range(trial_number):
        for tt in range(num_behav_trial):
        
            _,_,goal_state_arr[vv,mm,tt] = behavior_markov(transition_matrix[name][mm], total_time=total_time, start_state=0, end_state=[3,9], seed=init_seed+mm*100+tt*10+vv)
            
            if goal_state_arr[vv,mm,tt] == 3: goal_count[vv,0,mm] += 1; continue
            elif goal_state_arr[vv,mm,tt] == 9: goal_count[vv,1,mm] += 1; continue
            else: goal_count[vv,2,mm] += 1
    frac[name] = goal_count[vv,:,:].T/num_behav_trial

del goal_state_arr, goal_count

In [22]:
for name in MI_scenarios:
    print("%-8s P(food, water, neither) mean:" % name, frac[name].mean(0).round(3))


hungry   P(food, water, neither) mean: [0.58  0.328 0.092]
thirsty  P(food, water, neither) mean: [0.223 0.635 0.142]


In [23]:
from statsmodels.stats.multitest import multipletests

labels = ["Food (S4)", "Water (S10)", "Neither"]
box_colors = ["tomato", "steelblue", "lightgrey"]

fig, axs = plt.subplots(1, 2, figsize=(5, 3), sharey=True)
jitter = 0.06 * (np.random.default_rng(7).random(trial_number) - 0.5)

for ax, name in zip(axs, MI_scenarios):
    data = [frac[name][:, jj] for jj in range(3)]
    bp = ax.boxplot(data, positions=[0, 1, 2], widths=0.5, patch_artist=True, showfliers=False)
    for patch, c in zip(bp["boxes"], box_colors):
        patch.set_facecolor(c)
    for jj in range(3):
        ax.scatter(jj + jitter, frac[name][:, jj], color="black", s=15, alpha=0.6, zorder=3)
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels(labels, rotation=30)
    ax.set_ylim(0, 1)
    ax.set_title(name.capitalize())

axs[0].set_ylabel("P(outcome)")
fig.suptitle("Average final destination across %d behavior runs\n(potentiated network)" % num_behav_trial)
plt.tight_layout(); plt.show()

if save_plot: 
    plt.savefig(os.path.join(plot_path, "Carey__final_destination.svg"), format="svg", bbox_inches="tight")

# ── Stats: within each scenario, food vs. water, food vs. neither, water vs. neither (paired across trials) ──
pair_indices = [(0, 1), (0, 2), (1, 2)]
pair_labels  = ["Food vs. Water", "Food vs. Neither", "Water vs. Neither"]

for name in MI_scenarios:
    print(f"--- {name.capitalize()} ---")
    pvals, rows = [], []
    for i, j in pair_indices:
        x, y = frac[name][:, i], frac[name][:, j]
        t_stat, p_val = stats.ttest_rel(x, y)
        d_z = cohens_d_paired(x, y)
        ci_lo, ci_hi = ci_paired(x, y)
        pvals.append(p_val)
        rows.append((x.mean(), y.mean(), t_stat, p_val, d_z, ci_lo, ci_hi))
    _, pvals_fdr, _, _ = multipletests(pvals, method="fdr_bh")
    for lbl, (xm, ym, t_stat, p_val, d_z, ci_lo, ci_hi), p_fdr in zip(pair_labels, rows, pvals_fdr):
        print(f"  {lbl:18s} {xm:.3f} vs {ym:.3f}  t={t_stat:.3f}  p={p_val:.3g}  p_fdr={p_fdr:.3g}  "
              f"d_z={d_z:.2f}  CI=[{ci_lo:.2f}, {ci_hi:.2f}]")

--- Hungry ---
  Food vs. Water     0.580 vs 0.328  t=6.186  p=0.000162  p_fdr=0.000162  d_z=1.96  CI=[0.16, 0.34]
  Food vs. Neither   0.580 vs 0.092  t=30.290  p=2.28e-10  p_fdr=6.84e-10  d_z=9.58  CI=[0.45, 0.52]
  Water vs. Neither  0.328 vs 0.092  t=8.931  p=9.09e-06  p_fdr=1.36e-05  d_z=2.82  CI=[0.18, 0.30]
--- Thirsty ---
  Food vs. Water     0.223 vs 0.635  t=-20.554  p=7.13e-09  p_fdr=1.07e-08  d_z=-6.50  CI=[-0.46, -0.37]
  Food vs. Neither   0.223 vs 0.142  t=6.428  p=0.000121  p_fdr=0.000121  d_z=2.03  CI=[0.05, 0.11]
  Water vs. Neither  0.635 vs 0.142  t=30.568  p=2.1e-10  p_fdr=6.3e-10  d_z=9.67  CI=[0.46, 0.53]
